In [ ]:
#install llama-cpp-python on colab w gpu support, optional for gguf models
!CMAKE_ARGS="-DGGML_CUDA=on" pip install llama-cpp-python

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 50.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.5 MB/s eta 0:00:00
  Created wheel for llama-cpp-python: filename=llama_cpp_python-0.3.16-cp312-cp312-linux_x86_64.whl size=33389726 sha256=d85b6553a42cc131ad3014f6e620dd34724a4e90516765e00f986ca06618a272
  Stored in directory: /root/.cache/pip/wheels/90/82/ab/8784ee3fb99ddb07fd36a679ddbe63122cc07718f6c1eb3be8
Successfully built llama-cpp-python


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import time

# Google AI Studio imports
try:
    from google import genai
    from google.colab import userdata
    print("Google AI Studio libraries imported.")
except ImportError:
    genai = None
    print("Google AI Studio libraries not found. Run `pip install -U google-generativeai` if needed.")

# google drive import
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator

# Detect device
device = "cpu"
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"

print(f"Using device: {device}")

# Try to import llama_cpp for GGUF support
try:
    from llama_cpp import Llama
    print("llama-cpp-python is available.")
except ImportError:
    Llama = None
    print("llama-cpp-python is NOT installed. GGUF models will not be supported.")

Google AI Studio libraries imported.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/SP26Courses/CSE4061/ReadmeGenerator
Using device: cuda
llama-cpp-python is available.


Declaring the Quen3-Coder-Next Model and Loading the Tokenizer

In [ ]:
model_name = "unsloth/Qwen3-4B-Instruct-2507-GGUF"
# Example GGUF usage: model_name = "TheBloke/Llama-2-7B-Chat-GGUF"
# Example Gemini usage: model_name = "aistudio-gemini-1.5-flash"

tokenizer = None
model = None

if "gguf" in model_name.lower():
    if Llama is None:
        raise ImportError("You requested a GGUF model but llama-cpp-python is not installed. Please install it with `pip install llama-cpp-python`.")

    print(f"Loading GGUF model: {model_name} using llama-cpp-python...")
    # Attempt to load local path or download from Hub
    if os.path.exists(model_name):
        model = Llama(model_path=model_name, n_gpu_layers=-1, verbose=False, n_ctx=32768) # Added n_ctx
    else:
        # Downloads a generic quantized version if not specified
        model = Llama.from_pretrained(
            repo_id=model_name,
            filename="*Q4_0.gguf",
            verbose=False,
            n_gpu_layers=-1, # Offload to GPU if available
            n_ctx=80000 # Added n_ctx to utilize full context window
        )
    print("GGUF Model loaded successfully!")

elif model_name.startswith("aistudio-"):
    if genai is None:
         raise ImportError("Google Generative AI SDK not installed. Please install it.")

    real_model_name = model_name.replace("aistudio-", "")
    print(f"Configuring Google AI Studio model: {real_model_name}...")

    try:
        GOOGLE_API_KEY = userdata.get('google_aistudio_key')
        client = genai.Client(api_key=GOOGLE_API_KEY)
        print("Google AI Studio Model configured successfully!")
        try:
            for m in client.models.list():
                print(m.name)
        except Exception as e:
            print(f"Error listing models: {e}")
    except Exception as e:
        print(f"Error configuring Google AI Studio: {e}. Make sure 'google_aistudio_key' is set in Colab secrets.")
        model = None

else:
    # Standard Transformers loading
    print(f"Loading Transformers model: {model_name} on {device}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
    print("Transformers Model loaded successfully!")

Loading GGUF model: unsloth/Qwen3-4B-Instruct-2507-GGUF using llama-cpp-python...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
llama_context: n_ctx_per_seq (80000) < n_ctx_train (262144) -- the full capacity of the model will not be utilized


GGUF Model loaded successfully!


README Generation

In [ ]:
def generate_readme(parsed_file, model, tokenizer):
    # Check if the parsed file exists
    if not os.path.exists(parsed_file):
        print(f"File {parsed_file} does not exist.")
        return None

    with open(parsed_file, 'r', encoding='utf-8', errors='ignore') as f:
        code_content = f.read()

    # Create a prompt
    prompt = f"Generate a comprehensive README.md file based on the following repository code structure and contents:\n\n{code_content}\n\n"

    readme_content = ""

    # --- Inference Logic ---

    # 1. Google AI Studio (Gemini)
    # We check if 'client' is available in globals, which indicates the aistudio setup was run.
    if 'client' in globals() and client is not None and (model is None or hasattr(model, 'generate_content')):
        try:
            print(f"Generating content with Google AI Studio model: {real_model_name}...")
            # Using the updated client.models.generate_content syntax
            response = client.models.generate_content(model=f"models/{real_model_name}", contents=prompt)
            readme_content = response.text
        except Exception as e:
            print(f"Error generating content with Gemini: {e}")
            return ""

    # 2. llama-cpp-python
    elif model is not None and hasattr(model, 'create_completion'):
        print("Generating content with llama-cpp...")
        output = model.create_completion(
            prompt,
            max_tokens=4096,
            temperature=0.7,
            top_p=0.9,
            echo=False
        )
        readme_content = output['choices'][0]['text']

    # 3. Hugging Face Transformers
    elif model is not None and tokenizer is not None:
        print("Generating content with Transformers...")
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=16384  # Adjusted for potentially large context
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=4096,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=True,
                temperature=0.7,
                top_p=0.9
            )

        readme_content = tokenizer.decode(outputs[0], skip_special_tokens=True)

    else:
        print("Error: No valid model configuration or client detected.")
        return ""

    # Post-processing
    if "README.md" in readme_content:
        # Attempt to split to get content after the header if the model echoes it
        parts = readme_content.split("README.md", 1)
        if len(parts) > 1:
            readme_content = parts[1].strip()

    # Clean up code blocks if the model wrapped the output
    if readme_content.startswith("```markdown"):
        readme_content = readme_content.replace("```markdown", "", 1)
    elif readme_content.startswith("```"):
        readme_content = readme_content.replace("```", "", 1)

    if readme_content.endswith("```"):
        readme_content = readme_content[:-3]

    return readme_content.strip()

In [ ]:
def save_readme(output_path, readme_content, prefix=""):
    # Saves the generated README.md content to a file in the specified output path
    filename = f"{prefix}generated_README.md" if prefix else "generated_README.md"
    output_path = os.path.join(output_path, filename)

    with open(output_path, 'w', encoding='utf-8') as f:
        f.write(readme_content)

    print(f"{prefix}README.md has been saved to {output_path}")
    return output_path

In [ ]:
import os
import time # Ensure time is imported, though it was in a previous cell

def process_all_parsed_files(model, tokenizer, parsed_files_dir, output_path):
    print("Starting README generation for all parsed files...")

    # Ensure output directory exists
    if not os.path.exists(output_path):
        os.makedirs(output_path)

    # Iterate through all files in the parsed_files_dir
    for filename in os.listdir(parsed_files_dir):
        if filename.endswith("_parsed_code.txt"): # Assuming parsed files have this extension
            parsed_file_path = os.path.join(parsed_files_dir, filename)
            print(f"Processing: {parsed_file_path}")

            try:
                # Record start time
                time_start = time.time()

                # Generate the README
                readme_content = generate_readme(parsed_file_path, model, tokenizer)

                # Record end time
                time_end = time.time()
                time_elapsed = time_end - time_start

                if readme_content:
                    # Extract a prefix for the output filename (e.g., from 'apache_cordova-plugin-splashscreen_parsed_code.txt' to 'apache_cordova-plugin-splashscreen_')
                    prefix = filename.replace('_parsed_code.txt', '_')

                    # Print the generated README content and time taken
                    print("")
                    print(f"Generated README for {filename}:")
                    print(readme_content)
                    print(f"Time taken for {filename}: {time_elapsed:.2f} seconds")
                    print("")

                    # Save the generated README
                    save_readme(output_path, readme_content, prefix=prefix)
                else:
                    print(f"Skipping README generation for {filename} due to empty content.")
            except Exception as e:
                print(f"Error processing file {filename}: {e}")
    print("Finished README generation for all parsed files.")

In [ ]:
parsed_files_dir = "./out"
output_path = "llm_output"

process_all_parsed_files(model, tokenizer, parsed_files_dir, output_path)

Starting README generation for all parsed files...
Processing: ./out/apache_cordova-plugin-splashscreen_parsed_code.txt
Generating content with llama-cpp...

Generated README for apache_cordova-plugin-splashscreen_parsed_code.txt:
Size: 7220 bytes
Lines: 182
---
---
title: Browser Splashscreen
description: Control the browser platform splash screen for your app.
---
<!--
# license: Licensed to the Apache Software Foundation (ASF) under one
#         or more contributor license agreements.  See the NOTICE file
#         distributed with this work for additional information
#         regarding copyright ownership.  The ASF licenses this file
#         to you under the Apache License, Version 2.0 (the
#         "License"); you may not use this file except in compliance
#         with the License.  You may obtain a copy of the License at
#
#           http://www.apache.org/licenses/LICENSE-2.0
#
#         Unless required by applicable law or agreed to in writing,
#         software distrib